## Поиск аномалий (LOF и Isolation Forest) — версия для Google Colab

Данные (`banks.txt`, кодировка CP1251) загружаются с Google Drive автоматически. Сначала выбросы ищутся методом LOF при разных значениях порога contamination, затем аномалии удаляются и модель сохраняется. В конце тот же поиск повторяется методом Isolation Forest, который работает по другому принципу.


In [1]:
# Установка gdown и загрузка исходных данных с Google Drive.
# gdown скачивает публичный файл Google Drive по его ID.
# ID берётся из ссылки вида https://drive.google.com/file/d/<ID>/view
!pip install -q gdown
import gdown

FILE_ID = "18jJr3OcT6WY9YrbjGM9dl1VFOZw68qej"   # banks.txt
file_path = "/content/banks.txt"
gdown.download(id=FILE_ID, output=file_path, quiet=False)
print("Файл сохранён в:", file_path)


Downloading...
From: https://drive.google.com/uc?id=18jJr3OcT6WY9YrbjGM9dl1VFOZw68qej
To: /content/banks.txt
100%|██████████| 14.2k/14.2k [00:00<00:00, 20.0MB/s]

Файл сохранён в: /content/banks.txt


В поиске аномалий мы ищем выбросы в данных. Делаем мы это для того, чтобы избавиться от выбросов, которые будут своими экстремальными значениями влиять на качество данных и итоговую модель, которая будет слишком чустветильна к новым данным.

Сделаем поиск аномалий методом LOF. Метод выдаёт оценку "странности" (outlier score), которую потом нужно перевести в бинарный -1/+1.
Чтобы перевести оценку в метки (-1 = аномалия, 1 = норма), нужно пороговое значение. Для этого нужен параметр contamination (загрязнение), он помогает установить этот порог.
Чем выше contamination, тем больше объектов будет помечено как аномалии.

Это не гарантирует точное число аномалий, но задаёт ориентир. При contamination='auto' используется эвристика — может быть менее стабильно.

У метода LOF есть 2 основных режима работы: поиск аномалий и поиск новизны.

Поиск аномалий (novelty=False) (по умолчанию) - обнаружение выбросов в обучающих данных. На вход подаётся сырой датасет. Затем для каждой точки ищутся k ближайших соседей (n_neighbors). Считается локальная плотность: сколько рядом точек. Если у точки низкая плотность, а у её соседей — высокая, точка - она аномалия.
Вычисляется LOF-оценка:
LOF ≈ 1: точка в плотной области (норма)
LOF > 1: точка изолирована (возможно, аномалия)


Поиск новизны (novelty=True) - обнаружение новизны на очищенном датасете или на аналогичном по содержанию датасете с тем же распределением (например, данные за другой период времени). В этом режиме, когда уже убраны и известны выбросы, метод ищет новые точки на обновлённых данных и определяет, являются ли эти точки аномалиями относительно этого распределения.

In [2]:
# ПОИСК АНОМАЛИЙ методом LOF (Local Outlier Factor).
# Идея: у каждой точки оцениваем плотность её ближайших соседей. Если точка стоит
# в разреженной области, а соседи - в плотной, она считается выбросом.
# Параметр contamination задаёт ОЖИДАЕМУЮ долю аномалий (порог отсечения).
# Пробуем три значения (0.05, 0.10, 0.15) и сравниваем результаты.
# predict возвращает: -1 = аномалия, 1 = норма. Данные заранее стандартизируем.

# ============================================================
# ПОИСК АНОМАЛИЙ МЕТОДОМ LOF (Local Outlier Factor)
# ============================================================
# Файл: banks.txt (кодировка CP1251)
# Метод: LOF с novelty=False
# Contamination: 0.05, 0.10, 0.15
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
import warnings
warnings.filterwarnings("ignore")

# ---------- ЗАГРУЗКА ДАННЫХ ----------
file_path = "/content/banks.txt"

print("=" * 70)
print("ЗАГРУЗКА ФАЙЛА")
print("=" * 70)

# Пробуем разные разделители
df = None
for sep in ["\t", ";", ",", r"\s+"]:
    try:
        temp = pd.read_csv(file_path, sep=sep, encoding="cp1251", engine="python")
        if temp.shape[1] > 1:
            df = temp
            print(f"Файл успешно загружен (разделитель: '{sep}')")
            break
    except Exception as e:
        continue

if df is None:
    raise Exception("Не удалось прочитать файл")

print(f"Всего строк: {len(df)}")
print(f"Всего столбцов: {len(df.columns)}")
print(f"Столбцы: {list(df.columns)}")

# ---------- ПОИСК СТОЛБЦА С НАЗВАНИЯМИ БАНКОВ ----------
# Ищем текстовый столбец, который может содержать названия банков
name_col = None
for col in df.columns:
    if df[col].dtype == object:
        # Проверяем, что это действительно текстовые названия (не числа в строках)
        sample = df[col].dropna().astype(str).head(20)
        if sample.str.contains(r"[A-Za-zА-Яа-я]").any():
            name_col = col
            break

if name_col is None:
    # Если текстового столбца нет — используем индекс
    df["_bank_name_"] = ["Банк_" + str(i) for i in range(len(df))]
    name_col = "_bank_name_"
    print(" Текстовый столбец с названиями не найден, используются индексы")
else:
    print(f"Столбец с названиями банков: '{name_col}'")

# ---------- ОТБОР ЧИСЛОВЫХ ПРИЗНАКОВ ----------
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nЧисловых признаков для анализа: {len(numeric_cols)}")
print(f"   {numeric_cols}")

if len(numeric_cols) == 0:
    raise Exception("Не найдено числовых признаков для анализа")

X = df[numeric_cols].copy()

# Заполняем пропуски средними значениями
if X.isnull().any().any():
    print(" Обнаружены пропуски — заполняем средними значениями")
    X = X.fillna(X.mean())

# ---------- МАСШТАБИРОВАНИЕ ----------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(" Данные стандартизированы (StandardScaler)")

# ---------- ЗАПУСК LOF С РАЗНЫМИ CONTAMINATION ----------
contaminations = [0.05, 0.10, 0.15]
all_results = {}

print("\n" + "=" * 70)
print("ЗАПУСК АЛГОРИТМА LOF")
print("=" * 70)

for cont in contaminations:
    print(f"\n{'─' * 70}")
    print(f"Contamination = {cont}")
    print(f"{'─' * 70}")

    # novelty=False — обучение и предсказание на одних данных
    lof = LocalOutlierFactor(
        n_neighbors=20,
        contamination=cont,
        novelty=False,
        metric="minkowski"
    )

    # predictions: -1 = аномалия, 1 = норма
    predictions = lof.fit_predict(X_scaled)
    scores = lof.negative_outlier_factor_  # чем меньше — тем аномальнее

    n_anomalies = (predictions == -1).sum()
    n_normal = (predictions == 1).sum()

    print(f"   Всего объектов:        {len(predictions)}")
    print(f"   Нормальных объектов:   {n_normal}")
    print(f"   Найдено аномалий:      {n_anomalies}")
    print(f"   Доля аномалий:         {n_anomalies / len(predictions):.4f}")

    # Сохраняем результаты
    result_df = df[[name_col] + numeric_cols].copy()
    result_df["LOF_score"] = scores
    result_df["Anomaly"] = predictions
    result_df["Is_Anomaly"] = (predictions == -1)

    all_results[cont] = result_df

# ---------- СВОДНАЯ ТАБЛИЦА ВСЕХ АНОМАЛИЙ ----------
print("\n" + "=" * 70)
print("СВОДНАЯ ТАБЛИЦА АНОМАЛИЙ ПО ВСЕМ CONTAMINATION")
print("=" * 70)

# Собираем индексы аномалий по каждому contamination
anomaly_indices = set()
for cont, res in all_results.items():
    idx = res.index[res["Is_Anomaly"]].tolist()
    anomaly_indices.update(idx)

anomaly_indices = sorted(anomaly_indices)

# Формируем итоговую таблицу
summary = pd.DataFrame({
    "Банк": df.loc[anomaly_indices, name_col].values
}, index=anomaly_indices)

for cont in contaminations:
    col_name = f"Аномалия (cont={cont})"
    summary[col_name] = all_results[cont].loc[anomaly_indices, "Is_Anomaly"].map(
        {True: "ДА", False: "—"}
    )

# Добавим LOF-скоры для наглядности
for cont in contaminations:
    summary[f"LOF_score ({cont})"] = all_results[cont].loc[anomaly_indices, "LOF_score"].round(4)

# Сортируем по «средней аномальности» — чем чаще и сильнее, тем выше
summary["Сумма флагов"] = summary[[f"Аномалия (cont={c})" for c in contaminations]]\
    .apply(lambda row: (row == "ДА").sum(), axis=1)
summary = summary.sort_values("Сумма флагов", ascending=False)

print(f"\nУникальных объектов, попавших в аномалии хотя бы раз: {len(summary)}")

# Красивый вывод
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

print("\n" + summary.to_string())

# ---------- СТАТИСТИКА ПО СОВПАДЕНИЯМ ----------
print("\n" + "=" * 70)
print("СТАТИСТИКА СОВПАДЕНИЙ")
print("=" * 70)

for cont in contaminations:
    cnt = all_results[cont]["Is_Anomaly"].sum()
    print(f"   Contamination={cont}:  {cnt:3d} аномалий")

# Пересечение всех трёх
common_all = set(all_results[0.05].index[all_results[0.05]["Is_Anomaly"]])
for cont in contaminations:
    common_all &= set(all_results[cont].index[all_results[cont]["Is_Anomaly"]])
print(f"\n   Аномалии, найденные при ВСЕХ contamination: {len(common_all)}")
if len(common_all) > 0:
    print(f"      {df.loc[sorted(common_all), name_col].tolist()}")

# ---------- СОХРАНЕНИЕ РЕЗУЛЬТАТОВ ----------
output_path = "/content/anomalies_result.xlsx"
try:
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        summary.to_excel(writer, sheet_name="Сводка аномалий")
        for cont in contaminations:
            all_results[cont].to_excel(writer, sheet_name=f"cont_{cont}")
    print(f"\nРезультаты сохранены в: {output_path}")
except Exception as e:
    print(f"\n Не удалось сохранить Excel: {e}")

print("\nГОТОВО!")

ЗАГРУЗКА ФАЙЛА
Файл успешно загружен (разделитель: ',')
Всего строк: 371
Всего столбцов: 6
Столбцы: ['Bank', 'Assents', 'OwnCapital', 'IndFunds', 'NBSLoans', 'IndLoans']
Столбец с названиями банков: 'Bank'

Числовых признаков для анализа: 5
   ['Assents', 'OwnCapital', 'IndFunds', 'NBSLoans', 'IndLoans']
 Данные стандартизированы (StandardScaler)

ЗАПУСК АЛГОРИТМА LOF

──────────────────────────────────────────────────────────────────────
Contamination = 0.05
──────────────────────────────────────────────────────────────────────
   Всего объектов:        371
   Нормальных объектов:   352
   Найдено аномалий:      19
   Доля аномалий:         0.0512

──────────────────────────────────────────────────────────────────────
Contamination = 0.1
──────────────────────────────────────────────────────────────────────
   Всего объектов:        371
   Нормальных объектов:   334
   Найдено аномалий:      37
   Доля аномалий:         0.0997

─────────────────────────────────────────────────────────

Удаляем аномалии и сохраняем модель.

In [3]:
# УДАЛЕНИЕ найденных аномалий и обучение финальной модели LOF.
# Список anomalies - объекты, признанные выбросами на прошлом шаге. Убираем их,
# сохраняем очищенный файл, обучаем модель на чистых данных и сохраняем её вместе
# со scaler-ом (joblib), чтобы потом переиспользовать без повторного обучения.

# ============================================================
# УДАЛЕНИЕ АНОМАЛИЙ + СОХРАНЕНИЕ ФАЙЛА И МОДЕЛИ
# ============================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
import joblib
import os
import warnings
warnings.filterwarnings("ignore")

# ---------- ПУТИ ----------
base_dir = "/content"
file_path = os.path.join(base_dir, "banks.txt")
clean_path = os.path.join(base_dir, "banks_clean.txt")
model_path = os.path.join(base_dir, "lof_model.pkl")
scaler_path = os.path.join(base_dir, "lof_scaler.pkl")
cols_path = os.path.join(base_dir, "lof_columns.pkl")

# ---------- СПИСОК АНОМАЛИЙ ----------
anomalies = [
    '«Пересвет»', '«Траст»', '«ФК Открытие»', 'Альфа-банк', 'Балтинвестбанк',
    'БМ-банк', 'ВТБ', 'ВУЗ-банк', 'Газпромбанк', 'Генбанк',
    'Московский кредитный банк', 'Мособлбанк', 'МС Банк Рус', 'Почта-банк',
    'Промсвязьбанк', 'Райффайзенбанк', 'Россельхозбанк', 'Сбербанк России',
    'Севастопольский морской банк'
]

# ---------- ЗАГРУЗКА ----------
print("=" * 70)
print("ЗАГРУЗКА ИСХОДНОГО ФАЙЛА")
print("=" * 70)

df = None
for sep in ["\t", ";", ",", r"\s+"]:
    try:
        temp = pd.read_csv(file_path, sep=sep, encoding="cp1251", engine="python")
        if temp.shape[1] > 1:
            df = temp
            print(f"Загружено (разделитель '{sep}'): {len(df)} строк, {len(df.columns)} столбцов")
            break
    except Exception:
        continue

if df is None:
    raise Exception("Не удалось прочитать файл")

# ---------- ПОИСК СТОЛБЦА С НАЗВАНИЯМИ ----------
name_col = None
for col in df.columns:
    if df[col].dtype == object:
        sample = df[col].dropna().astype(str).head(20)
        if sample.str.contains(r"[A-Za-zА-Яа-я]").any():
            name_col = col
            break

if name_col is None:
    raise Exception("Не найден столбец с названиями банков")

print(f"Столбец с названиями: '{name_col}'")

# ---------- УДАЛЕНИЕ АНОМАЛИЙ ----------
print("\n" + "=" * 70)
print("УДАЛЕНИЕ АНОМАЛИЙ")
print("=" * 70)

before = len(df)
mask = ~df[name_col].astype(str).str.strip().isin([a.strip() for a in anomalies])
df_clean = df[mask].reset_index(drop=True)
removed = before - len(df_clean)

# Показать, кто реально был удалён
found = df[~mask][name_col].astype(str).tolist()
missing = [a for a in anomalies if a not in found]

print(f"   Было объектов:    {before}")
print(f"   Осталось:         {len(df_clean)}")
print(f"    Удалено:          {removed}")

if missing:
    print(f"\n    Не найдены в файле (проверьте написание):")
    for m in missing:
        print(f"      • {m}")

# ---------- СОХРАНЕНИЕ ОЧИЩЕННОГО ФАЙЛА ----------
print("\n" + "=" * 70)
print("СОХРАНЕНИЕ ОЧИЩЕННОГО ФАЙЛА (CP1251)")
print("=" * 70)

# Определяем разделитель исходного файла
with open(file_path, "r", encoding="cp1251") as f:
    first_line = f.readline()
if "\t" in first_line:
    sep_out = "\t"
elif ";" in first_line:
    sep_out = ";"
elif "," in first_line:
    sep_out = ","
else:
    sep_out = "\t"

df_clean.to_csv(clean_path, sep=sep_out, index=False, encoding="cp1251")
print(f"   Сохранено: {clean_path}")
print(f"   Строк: {len(df_clean)}, разделитель: '{sep_out}', кодировка: cp1251")

# ---------- ОБУЧЕНИЕ ФИНАЛЬНОЙ МОДЕЛИ LOF ----------
print("\n" + "=" * 70)
print("ОБУЧЕНИЕ МОДЕЛИ LOF НА ОЧИЩЕННЫХ ДАННЫХ")
print("=" * 70)

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
print(f"   Признаков: {len(numeric_cols)}")
print(f"      {numeric_cols}")

X = df_clean[numeric_cols].copy()
if X.isnull().any().any():
    X = X.fillna(X.mean())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Финальный contamination — можно выбрать средний, например 0.10
FINAL_CONT = 0.10
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=FINAL_CONT,
    novelty=False,
    metric="minkowski"
)
lof.fit(X_scaled)

# Проверка на очищенных данных
preds = lof.fit_predict(X_scaled)
n_anom = (preds == -1).sum()
print(f"   Модель обучена (contamination={FINAL_CONT})")
print(f"   Остаточных аномалий в очищенном датасете: {n_anom}")

# ---------- СОХРАНЕНИЕ МОДЕЛИ ----------
print("\n" + "=" * 70)
print("СОХРАНЕНИЕ МОДЕЛИ")
print("=" * 70)

joblib.dump(lof, model_path)
joblib.dump(scaler, scaler_path)
joblib.dump(numeric_cols, cols_path)

print(f"   Модель LOF:      {model_path}")
print(f"   Scaler:          {scaler_path}")
print(f"   Список колонок:  {cols_path}")

# ---------- ПРИМЕР ЗАГРУЗКИ МОДЕЛИ ----------
print("\n" + "=" * 70)
print("ПРИМЕР ИСПОЛЬЗОВАНИЯ СОХРАНЁННОЙ МОДЕЛИ")
print("=" * 70)

example_code = '''
import joblib, pandas as pd
from sklearn.neighbors import LocalOutlierFactor

lof = joblib.load(r"''' + model_path + '''")
scaler = joblib.load(r"''' + scaler_path + '''")
cols = joblib.load(r"''' + cols_path + '''")

# Для нового объекта нужен метод novelty=True, либо:
# переобучить с novelty=True. Здесь показан вариант проверки через decision_function
# (доступно только при novelty=True). Для novelty=False используйте negative_outlier_factor_
'''
print(example_code)

print("ГОТОВО!")

ЗАГРУЗКА ИСХОДНОГО ФАЙЛА
Загружено (разделитель ','): 371 строк, 6 столбцов
Столбец с названиями: 'Bank'

УДАЛЕНИЕ АНОМАЛИЙ
   Было объектов:    371
   Осталось:         352
    Удалено:          19

СОХРАНЕНИЕ ОЧИЩЕННОГО ФАЙЛА (CP1251)
   Сохранено: /content/banks_clean.txt
   Строк: 352, разделитель: ',', кодировка: cp1251

ОБУЧЕНИЕ МОДЕЛИ LOF НА ОЧИЩЕННЫХ ДАННЫХ
   Признаков: 5
      ['Assents', 'OwnCapital', 'IndFunds', 'NBSLoans', 'IndLoans']
   Модель обучена (contamination=0.1)
   Остаточных аномалий в очищенном датасете: 36

СОХРАНЕНИЕ МОДЕЛИ
   Модель LOF:      /content/lof_model.pkl
   Scaler:          /content/lof_scaler.pkl
   Список колонок:  /content/lof_columns.pkl

ПРИМЕР ИСПОЛЬЗОВАНИЯ СОХРАНЁННОЙ МОДЕЛИ

import joblib, pandas as pd
from sklearn.neighbors import LocalOutlierFactor

lof = joblib.load(r"/content/lof_model.pkl")
scaler = joblib.load(r"/content/lof_scaler.pkl")
cols = joblib.load(r"/content/lof_columns.pkl")

# Для нового объекта нужен метод novelty=True, л

Теперь прогоним сырой датасет через обученную модель, но уже с novelty=True. То есть будем искать эти самые аномамлии.

In [4]:
# РЕЖИМ novelty=True: модель обучается на ОЧИЩЕННЫХ данных, а затем ищет аномалии
# в исходных данных относительно "нормального" распределения.
# decision_function < 0 указывает на аномалию. В конце сравниваем, кого нашла
# модель, с тем, кого мы удаляли вручную ранее.

# ============================================================
# ПРОГОН ИСХОДНОГО ФАЙЛА ЧЕРЕЗ МОДЕЛЬ LOF (novelty=True)
# ============================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
import joblib
import os
import warnings
warnings.filterwarnings("ignore")

# ---------- ПУТИ ----------
base_dir = "/content"
file_path = os.path.join(base_dir, "banks.txt")
clean_path = os.path.join(base_dir, "banks_clean.txt")
model_path = os.path.join(base_dir, "lof_model.pkl")
scaler_path = os.path.join(base_dir, "lof_scaler.pkl")
cols_path = os.path.join(base_dir, "lof_columns.pkl")
report_path = os.path.join(base_dir, "lof_novelty_report.xlsx")

# ---------- ЗАГРУЗКА ФАЙЛОВ ----------
print("=" * 70)
print("ЗАГРУЗКА ДАННЫХ И АРТЕФАКТОВ")
print("=" * 70)

def load_csv(path):
    for sep in ["\t", ";", ",", r"\s+"]:
        try:
            temp = pd.read_csv(path, sep=sep, encoding="cp1251", engine="python")
            if temp.shape[1] > 1:
                return temp
        except Exception:
            continue
    raise Exception(f"Не удалось прочитать {path}")

df_original = load_csv(file_path)   # исходный файл (со всеми банками)
df_clean    = load_csv(clean_path)  # очищенный файл (без аномалий)

# Загружаем scaler и список колонок
scaler = joblib.load(scaler_path)
cols = joblib.load(cols_path)

print(f"Исходный файл:  {len(df_original)} строк")
print(f"Очищенный файл: {len(df_clean)} строк")
print(f"Признаки ({len(cols)}): {cols}")

# ---------- ПОИСК СТОЛБЦА С НАЗВАНИЯМИ ----------
name_col = None
for col in df_original.columns:
    if df_original[col].dtype == object:
        sample = df_original[col].dropna().astype(str).head(20)
        if sample.str.contains(r"[A-Za-zА-Яа-я]").any():
            name_col = col
            break
if name_col is None:
    raise Exception("Не найден столбец с названиями банков")
print(f"Столбец с названиями: '{name_col}'")

# ---------- ПЕРЕОБУЧЕНИЕ МОДЕЛИ С novelty=True ----------
print("\n" + "=" * 70)
print("ПЕРЕОБУЧЕНИЕ LOF С novelty=True (на очищенных данных)")
print("=" * 70)

X_clean = df_clean[cols].copy().fillna(df_clean[cols].mean())
X_clean_scaled = scaler.transform(X_clean)

lof_novelty = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.10,   # можно поменять
    novelty=True,         # ключевое отличие
    metric="minkowski"
)
lof_novelty.fit(X_clean_scaled)
print("Модель обучена и готова к предсказаниям на новых данных")

# ---------- ПРОГОН ИСХОДНОГО ФАЙЛА ----------
print("\n" + "=" * 70)
print("ПРОГОН ИСХОДНОГО ФАЙЛА ЧЕРЕЗ МОДЕЛЬ")
print("=" * 70)

X_orig = df_original[cols].copy().fillna(df_original[cols].mean())
X_orig_scaled = scaler.transform(X_orig)

predictions = lof_novelty.predict(X_orig_scaled)                 # -1 или 1
scores      = lof_novelty.decision_function(X_orig_scaled)       # <0 аномалия
neg_scores  = lof_novelty.negative_outlier_factor_               # чем меньше, тем аномальнее (для train)

n_anomalies = (predictions == -1).sum()
n_normal    = (predictions == 1).sum()

print(f"   Всего объектов:     {len(predictions)}")
print(f"   Нормальных:         {n_normal}")
print(f"   Аномалий:           {n_anomalies}")
print(f"   Доля аномалий:      {n_anomalies / len(predictions):.4f}")

# ---------- ИТОГОВАЯ ТАБЛИЦА ----------
result = pd.DataFrame({
    "Банк": df_original[name_col].astype(str).values,
    "LOF_score": scores.round(4),
    "Предсказание": ["АНОМАЛИЯ" if p == -1 else "НОРМА" for p in predictions],
    "Is_Anomaly": predictions == -1,
})

# Сортируем: сначала аномалии, потом по возрастанию score
result = result.sort_values(["Is_Anomaly", "LOF_score"], ascending=[False, True]).reset_index(drop=True)

print("\n" + "=" * 70)
print("РЕЗУЛЬТАТЫ (все объекты, отсортированы: аномалии сверху)")
print("=" * 70)
print(result.to_string(index=False))

# ---------- ТОЛЬКО АНОМАЛИИ ----------
print("\n" + "=" * 70)
print("НАЙДЕННЫЕ АНОМАЛИИ")
print("=" * 70)
anomalies_only = result[result["Is_Anomaly"]][["Банк", "LOF_score", "Предсказание"]]
print(anomalies_only.to_string(index=False))

# ---------- СРАВНЕНИЕ С ОЧИЩЕННЫМ СПИСКОМ ----------
removed_names = set(df_original[name_col].astype(str)) - set(df_clean[name_col].astype(str))
found_names   = set(anomalies_only["Банк"])
print("\n" + "=" * 70)
print("СРАВНЕНИЕ: КОГО УДАЛЯЛИ vs КОГО НАШЛА МОДЕЛЬ")
print("=" * 70)
print(f"    Были удалены ранее:      {len(removed_names)}")
print(f"   Модель нашла:             {len(found_names)}")
print(f"   Совпало:                  {len(removed_names & found_names)}")
print(f"    Модель НЕ нашла (были удалены): {sorted(removed_names - found_names)}")
print(f"   Модель нашла сверх того:  {sorted(found_names - removed_names)}")

# ---------- СОХРАНЕНИЕ ОТЧЁТА ----------
try:
    with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
        result.to_excel(writer, sheet_name="Все объекты", index=False)
        anomalies_only.to_excel(writer, sheet_name="Только аномалии", index=False)
    print(f"\nОтчёт сохранён: {report_path}")
except Exception as e:
    print(f"\n Не удалось сохранить Excel: {e}")

# ---------- СОХРАНЕНИЕ novelty-МОДЕЛИ ----------
novelty_model_path = os.path.join(base_dir, "lof_model_novelty.pkl")
joblib.dump(lof_novelty, novelty_model_path)
print(f"novelty-модель сохранена: {novelty_model_path}")

print("\nГОТОВО!")

ЗАГРУЗКА ДАННЫХ И АРТЕФАКТОВ
Исходный файл:  371 строк
Очищенный файл: 352 строк
Признаки (5): ['Assents', 'OwnCapital', 'IndFunds', 'NBSLoans', 'IndLoans']
Столбец с названиями: 'Bank'

ПЕРЕОБУЧЕНИЕ LOF С novelty=True (на очищенных данных)
Модель обучена и готова к предсказаниям на новых данных

ПРОГОН ИСХОДНОГО ФАЙЛА ЧЕРЕЗ МОДЕЛЬ
   Всего объектов:     371
   Нормальных:         322
   Аномалий:           49
   Доля аномалий:      0.1321

РЕЗУЛЬТАТЫ (все объекты, отсортированы: аномалии сверху)
                        Банк  LOF_score Предсказание  Is_Anomaly
                     «Траст» -6070.7910     АНОМАЛИЯ        True
             Сбербанк России  -121.2426     АНОМАЛИЯ        True
                         ВТБ   -47.2370     АНОМАЛИЯ        True
                  Мособлбанк   -41.1194     АНОМАЛИЯ        True
              Балтинвестбанк   -19.2514     АНОМАЛИЯ        True
                 Газпромбанк   -17.0779     АНОМАЛИЯ        True
                  Альфа-банк   -10.4813    

Isolation forest работает по-другому. Он не измеряет плотность или расстояния, а изолирует точки случайными разрезами. Аномалии изолируются быстрее (за меньшее число разрезов), чем нормальные точки.

In [5]:
# АЛЬТЕРНАТИВНЫЙ метод: Isolation Forest.
# Он не считает расстояния или плотность, а строит случайные деревья, которые
# "разрезают" пространство. Аномалию удаётся изолировать за меньшее число разрезов,
# чем обычную точку, - на этом и основана оценка аномальности.
# Ниже: находим и удаляем аномалии, затем обучаем модель на чистых данных.

import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# --- Путь к файлу ---
file_path = "/content/banks.txt"

# --- 1. Загрузка данных ---
print("Загрузка banks.txt...")
try:
    df = pd.read_csv(file_path, sep=',', encoding='cp1251')
except UnicodeDecodeError:
    df = pd.read_csv(file_path, sep=',', encoding='utf-8')

bank_name_col = df.columns[0]
X = df.select_dtypes(include=[np.number])

if X.shape[1] == 0:
    raise ValueError("Нет числовых столбцов")

print(f"Загружено: {len(df)} банков")
print(f"Используется: '{bank_name_col}' как идентификатор")

# --- 2. Стандартизация ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ===========================================================================
# ШАГ 1: Поиск и удаление аномалий (аналог novelty=False)
# ===========================================================================
print("\nШАГ 1: Поиск аномалий с помощью Isolation Forest...")

iso_clean = IsolationForest(
    n_estimators=100,
    contamination=0.1,
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)
labels = iso_clean.fit_predict(X_scaled)  # -1 = аномалия, 1 = норма

# Фильтруем: оставляем только нормальные объекты
mask_normal = labels == 1
X_clean = X[mask_normal].copy()
df_clean = df[mask_normal].reset_index(drop=True)

print(f"Удалено как аномалии: {len(df) - len(df_clean)}")
print(f"   Осталось (чистые данные): {len(df_clean)}")

# ===========================================================================
# ШАГ 2: Обучение модели на чистых данных (аналог novelty=True)
# ===========================================================================
print("\nШАГ 2: Обучение Isolation Forest на чистых данных (мониторинг) ...")

# Масштабируем чистые данные
X_clean_scaled = scaler.fit_transform(X_clean)

# Обучаем "продовую" модель ТОЛЬКО на норме
iso_novelty = IsolationForest(
    n_estimators=100,
    contamination=0.1,           # можно оставить или потом использовать decision_function
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)
iso_novelty.fit(X_clean_scaled)

print("Модель Isolation Forest обучена на очищенных данных")

# ===========================================================================
# ШАГ 3: Прогноз на новых (чистых) данных — проверка работоспособности
# ===========================================================================
print("\nШАГ 3: Проверка 'новых' данных (например, новые банки)")

# Берём первые 5 из ОЧИЩЕННЫХ — как будто они пришли сегодня
X_new_sample = X_clean.iloc[:5].values
X_new_scaled = scaler.transform(X_new_sample)

# Предсказание
y_pred_new = iso_novelty.predict(X_new_scaled)
scores_new = iso_novelty.decision_function(X_new_scaled)

print("\nРезультаты для новых (чистых) банков:")
for i, idx in enumerate(df_clean.index[:5]):
    bank_name = df_clean.loc[idx, bank_name_col]
    pred = "Норма" if y_pred_new[i] == 1 else "Аномалия"
    score = scores_new[i]
    print(f"   • {bank_name:<30} | {pred} | Score: {score:.3f}")

# --- Дополнительно: проверим, не помечает ли она чистые данные как аномалии
n_anom_in_clean = np.sum(y_pred_new == -1)
if n_anom_in_clean == 0:
    print("\nОтлично: все чистые объекты распознаны как нормальные")
else:
    print(f"\n {n_anom_in_clean} из 5 помечены как аномалии — возможно, чувствительность высока")

# ===========================================================================
# ШАГ 4: Сохранение модели и скейлера
# ===========================================================================
import pickle

model_path = file_path.replace(".txt", "_isolation_forest_novelty.pkl")
scaler_path = file_path.replace(".txt", "_scaler.pkl")

with open(model_path, 'wb') as f:
    pickle.dump(iso_novelty, f)
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(f"\nМодель сохранена: {model_path}")
print(f"Scaler сохранён:   {scaler_path}")


Загрузка banks.txt...
Загружено: 371 банков
Используется: 'Bank' как идентификатор

ШАГ 1: Поиск аномалий с помощью Isolation Forest...
Удалено как аномалии: 37
   Осталось (чистые данные): 334

ШАГ 2: Обучение Isolation Forest на чистых данных (мониторинг) ...
Модель Isolation Forest обучена на очищенных данных

ШАГ 3: Проверка 'новых' данных (например, новые банки)

Результаты для новых (чистых) банков:
   • «Авангард»                     | Аномалия | Score: -0.004
   • «Аверс»                        | Аномалия | Score: -0.016
   • «Агора»                        | Норма | Score: 0.203
   • «Агропромкредит»               | Норма | Score: 0.149
   • «Агророс»                      | Норма | Score: 0.182

 2 из 5 помечены как аномалии — возможно, чувствительность высока

Модель сохранена: /content/banks_isolation_forest_novelty.pkl
Scaler сохранён:   /content/banks_scaler.pkl


Есть набор точек на плоскости.
Нормальные точки сконцентрированы в одном районе.
Аномалия — одна точка далеко в пустыне.
Чтобы отделить аномалию — нужно всего несколько разрезов.
Чтобы точно определить, где находится нормальный объект среди тысяч других — нужно много делений.

Это и лежит в основе метода Isolation Forest:
чем быстрее объект "изолируется", тем вероятнее он — аномалия.

1. Берётся случайная подвыборка данных (например, 256 объектов)
2. Дерево строится случайно:
Выбирается случайный признак (например, Сумма)
Выбирается случайное значение в диапазоне этого признака
Делится выборка: меньше / больше значения
3. Процесс повторяется для каждой подгруппы — пока:
Все объекты в листе одинаковы, или
Достигнута максимальная глубина

Одно дерево может ошибаться из-за случайности.

Поэтому:

Строится много деревьев
Для каждого объекта считается средняя глубина изоляции
Чем ниже средняя глубина тем выше шанс, что это аномалия

